In [1]:
from PIL import Image, ImageDraw, ImageFont
import requests
from io import BytesIO
import re
from bs4 import BeautifulSoup 
import os
import pandas as pd
from pathlib import Path

# Input

In [2]:
season_code = "25_26"
player_name = "Romain Perraud"
output_path = f"data/{player_name}_match.png"
current_dir = Path.cwd()  
path_folder = current_dir.parent / "csv" / f"csv{season_code}" / "players"

# Code Canva

In [3]:
def image_from_url(url):
    resp = requests.get(url)
    return Image.open(BytesIO(resp.content)).convert("RGBA")

def generate_list_data_from_percentiles(percentiles):
    max_width=580
    list_data = []
    for p in percentiles:
        length = max_width * (p / 100) 
        list_data.append((length, str(int(p))))
    return list_data

In [4]:
def generate_player_canva(joueur_info, data_names, percentiles):
    base = Image.open("data/canva_season.png").convert("RGBA")
    draw = ImageDraw.Draw(base)

    try:
        font_name = ImageFont.truetype("Arial.ttf", 40)  
        font_stat = ImageFont.truetype("Arial.ttf", 30)
        font_sec = ImageFont.truetype("Arial.ttf", 24)
    except OSError:
        font_name = ImageFont.load_default()
        font_stat = ImageFont.load_default()
        font_sec = ImageFont.load_default()

    # -----------------------
    # Photo joueur
    # -----------------------
    im_size = 191 
    im_x, im_y = 75, 73
    corner_radius = 20

    mask = Image.new("L", (im_size, im_size), 0)
    mask_draw = ImageDraw.Draw(mask)
    mask_draw.rounded_rectangle([(0, 0), (im_size, im_size)], radius=corner_radius, fill=255)
    player_img = image_from_url(joueur_info["image_url"])
    player_img = player_img.resize((im_size, im_size))
    player_img.putalpha(mask)
    base.paste(player_img, (im_x, im_y), mask=player_img)
    
    # -----------------------
    # Bloc nom du joueur
    # -----------------------
    draw.text(
        (355, 108),
        joueur_info['name'],
        fill="white",
        font=font_name,
        anchor="lm"  
    )

    # -----------------------
    # Drapeau et pays
    # -----------------------
    flag = image_from_url(joueur_info["flag_url"])
    flag_w, flag_h= 49, 35
    flag = flag.resize((flag_w, flag_h))
    y_mid = 184
    base.paste(flag, (355, int(y_mid - flag_h/2)), mask=flag)

    draw.text(
        (412, 183),
        joueur_info["nationality"],
        fill="white",
        font_size="15px", 
        font=font_stat,
        anchor="lm"
    )
    
    club_flag = image_from_url(joueur_info["club_url"])
    club_flag_w, club_flag_h = 45, 45  
    club_flag = club_flag.resize((club_flag_w, club_flag_h))
    base.paste(club_flag, (355, 205), mask=club_flag)

    draw.text(
        (412, 230),
        joueur_info["club"],
        fill="white",
        font=font_stat,
        anchor="lm"
    )

    # -----------------------
    # Barres centiles
    # -----------------------
    list_data = generate_list_data_from_percentiles(percentiles)
    start_x, start_y = 355, 310
    bar_height = 20
    bar_spacing = 77.6

    for i, (length, text) in enumerate(list_data):
        y = start_y + i * bar_spacing
        stat_name = data_names[i]

        draw.text((start_x, y - 25), stat_name, fill="white", font=font_stat, anchor="lm")
        draw.rectangle([(start_x, y), (start_x + length, y + bar_height)], fill=(241, 161, 159))
        draw.text((start_x + length + 10, y + bar_height/2), text, fill="white", font=font_stat, anchor="lm", align="center")

    
    # -----------------------
    # Stats secondaires
    # -----------------------
    sec_start_x, sec_start_y = 79, 320
    sec_spacing = 95
    for i, value in enumerate(joueur_info["stats_secondaires"]):
        y = sec_start_y + i * sec_spacing
        draw.text((sec_start_x, y), value, fill="white", font=font_sec)

    # -----------------------
    # Sauvegarde
    # -----------------------
    return base

# Code Transfermarkt

In [5]:
def new_date(date_str):
    _, mois, annee = date_str.split("/")
    mois = int(mois)
    annee = int(annee) 
    mois_fr = {
        1: "JANVIER",
        2: "FÉVRIER",
        3: "MARS",
        4: "AVRIL",
        5: "MAI",
        6: "JUIN",
        7: "JUILLET",
        8: "AOÛT",
        9: "SEPTEMBRRE",
        10: "OCTOBRE",
        11: "NOVEMBRE",
        12: "DÉCEMBRE"
    }
    
    return f"{mois_fr[mois]} {annee}"

In [6]:
def new_foot(foot):
    mapping = {
        "right": "DROIT",
        "left": "GAUCHE",
        "both": "AMBIDEXTRE"
    }
    return mapping.get(foot.lower(), foot)

In [7]:
def new_value(price_str):
    s = price_str.strip().replace("€", "").lower()

    if s.endswith("k"):
        value = s[:-1]  
        return f"{float(value):g} 000 €"

    if s.endswith("m"):
        value = s[:-1]  
        return f"{float(value):g} MILLIONS €"

    return f"{s} €"


In [8]:
def new_name(full_name):
    parts = full_name.strip().split()

    if len(parts) == 1: 
        return parts[0].capitalize()

    first_name = " ".join(parts[:-1])          
    last_name = parts[-1].upper()        

    return f"{first_name} {last_name}"


In [9]:
def new_age(text):
    m = re.search(r"\((\d+)\)", text)
    if m:
        age = int(m.group(1))
        return f"{age} ANS"
    return None


In [10]:
def new_position(text):
    if " - " in text:
        main_pos = text.split(" - ")[0].strip()
    else:
        main_pos = text.strip()
    
    if main_pos == "Attack":
        main_pos_new = "ATTAQUANT"
    elif main_pos == "Midfield":
        main_pos_new = "MILIEU"
    elif main_pos == "Defender":
        main_pos_new = "DÉFENSEUR"
    elif main_pos == "Goalkeeper":
        main_pos_new = "GARDIEN"
    
    return main_pos_new

In [11]:
class PlayerProfileScraper:
    def __init__(self, full_name):
        self.full_name = full_name
        self.full_name_for_url = full_name.replace(' ', '+')
        self.base_url = f"https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query={self.full_name_for_url}"
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
        }

    def fetch_search_results(self):
        response = requests.get(self.base_url, headers=self.headers)
        if response.status_code != 200:
            return None
        return response.text

    def parse_profile_url(self, html_content):
        soup = BeautifulSoup(html_content, 'html.parser')
        table = soup.find('table', class_='items')
        
        if table is None:
            return None
        
        first_row = table.find('tbody').find('tr')
        if first_row is None:
            return None
        
        player_link = first_row.find('td', class_='hauptlink').find('a', href=True)
        profile_url = "https://www.transfermarkt.com" + player_link['href']
        return profile_url


    def scrape_profile_info(self, profile_url):
        response = requests.get(profile_url, headers=self.headers)
        if response.status_code != 200:
            return None
        
        soup = BeautifulSoup(response.text, 'html.parser')

        h1_tag = soup.find('h1', class_='data-header__headline-wrapper')

        if h1_tag:
            first_names = "".join(t.strip() for t in h1_tag.find_all(string=True, recursive=False))
            last_name_tag = h1_tag.find('strong')
            last_name = last_name_tag.get_text(strip=True) if last_name_tag else ""
            name = f"{first_names} {last_name}".strip()

        market_value_tag = soup.find("a", class_="data-header__market-value-wrapper")
        market_value = None
        if market_value_tag:
            value_parts = [
                t.strip()
                for t in market_value_tag.stripped_strings
                if not t.startswith("Last update")
            ]
            market_value = "".join(value_parts)

        og_image_meta = soup.find('meta', property="og:image")
        image_url = og_image_meta['content'] if og_image_meta else None

        info = {}
        info_table_spans = soup.select("div.info-table span.info-table__content")

        i = 0
        while i < len(info_table_spans) - 1:
            label_span = info_table_spans[i]
            value_span = info_table_spans[i + 1]
            #print(label_span, value_span)
            if "info-table__content--regular" in label_span.get("class", []) and \
            "info-table__content--bold" in value_span.get("class", []):

                label = label_span.get_text(strip=True).rstrip(":")
                value = value_span.get_text(" ", strip=True)  
                info[label] = value
                i += 2
            else:
                i += 1

        height = info.get("Height")
        foot = info.get("Foot")
        contract_expires = info.get("Contract expires")
        club = info.get("Current club")
        age = info.get("Date of birth/Age")
        position = info.get("Position")

        citizenship_tag = soup.find("span", itemprop="nationality")
        nationality = None
        flag_url = None
        if citizenship_tag:
            nationality = citizenship_tag.get_text(strip=True)
            flag_img = citizenship_tag.find("img")
            flag_url = flag_img["src"] if flag_img else None

        club_url = None 
        club_name = club.lower()
        club_name = re.sub(r"[^a-z0-9]+", "-", club_name)
        club_name = club_name.strip("-")
        club_a_tag = soup.find("a", href=lambda x: x and club_name.lower() in x.lower())
        if club_a_tag:
            img_tag = club_a_tag.find("img")
            if img_tag:
                srcset = img_tag.get("srcset", "")
                club_url = srcset.split(",")[0].split()[0]
        
        return {
            "name": new_name(name),
            "nationality": nationality.upper(),
            "club": club.upper(),
            "club_url": club_url,
            "image_url": image_url,
            "flag_url": flag_url,
            "stats_secondaires": ["", "", "", new_date(contract_expires), new_foot(foot), new_position(position), new_age(age), new_value(market_value)]
        }

    def download_image(self, image_url, file_name):
        image_response = requests.get(image_url)
        if image_response.status_code == 200:
            with open(file_name, 'wb') as file:
                file.write(image_response.content)

    def save_player_profile(self):
        html_content = self.fetch_search_results()
        if html_content:
            profile_url = self.parse_profile_url(html_content)
            if profile_url:
                profile_info = self.scrape_profile_info(profile_url)
                if profile_info:
                    return profile_info
        return None

# Code Stats

In [12]:
def get_features_centiles(position):
    if position == 'ATTAQUANT':
        features_eng = [
            'Goals', 'Efficiency', '% Take-Ons', 'Actions created',
            'Expected Assists (xA)', "Actions in the Penalty Area", 'Key Passes',
            '% Aerial Duels', 'Progressive Actions (Total)', 'Successful Take-Ons'
        ]
        features_fr = [
            "MARQUER DES BUTS", "EFFICACITÉ", "% DRIBBLES RÉUSSIS", "CRÉER DES ACTIONS",
            "PASSES DÉCISIVES ATTENDUES (xA)", "ACTIONS DANS LA SURFACE", "PASSES CLÉS",
            "% DUELS AÉRIENS", "ACTIONS PROGRESSIVES (TOTAL)", "DRIBBLES RÉUSSIS"
        ]
    elif position == 'MILIEU':
        features_eng = [
            'Progressive Actions (Total)', 'Interceptions', 'Tackles Won',
            'Blocks', 'Ball Recoveries', 'Key Passes',
            'Fouls Committed', '% Tackles/Duels', 'Touches in Middle Third', '% Aerial Duels'
        ]
        features_fr = [
            "ACTIONS PROGRESSIVES (TOTAL)", "INTERCEPTIONS", "NBR TACLES RÉUSSIS",
            "CONTRES", "BALLONS RÉCUPÉRÉS", "PASSES CLÉS",
            "FAUTES COMMISES", "% TACLES / DUELS", "TOUCHES DANS LE TIERS CENTRAL",
            "% DUELS AÉRIENS"
        ]
    elif position == 'DÉFENSEUR':
        features_eng = [
            'Clearances', 'Blocks', 'Interceptions', '% Aerial Duels',
            'Touches', 'Fouls Committed', 'Aerials Won',
            'Progressive Passes', 'Ball Recoveries', '% Tackles/Duels'
        ]    
        features_fr = [
            "DÉGAGEMENTS", "CONTRES", "INTERCEPTIONS", "% DUELS AÉRIENS",
            "TOUCHES DE BALLE", "FAUTES COMMISES", "DUELS AÉRIENS GAGNÉS",
            "PASSES PROGRESSIVES", "BALLONS RÉCUPÉRÉS", "% TACLES / DUELS JOUÉS"
        ]
    elif position == "GARDIEN":
        features_eng = [
            "Clean Sheets", "Crosses Stopped", "Defensive Actions Outside Penalty Area", 
            "Saves", "Goals Against", "Save Efficiency", "% Saves", 
            "% Long Passes", "% Crosses Stopped", "PSxG/Save"
        ]
        features_fr = [
            "CLEAN SHEETS", "CAPTER LES CENTRES", "ACTIONS DÉFENSIVES HORS SURFACE", 
            "RÉALISER DES ARRÊTS", "NE PAS CONCÉDER DE BUTS", "EFFICACITÉ DES ARRÊTS", 
            "% ARRÊTS", "% PASSES LONGUES RÉUSSIES", "% CENTRES CAPTÉS", "DIFFICULTÉ DES ARRÊTS"
        ]
    return features_eng, features_fr

In [13]:
def get_data_centiles(player_name, position, features_centiles, path_folder):
    file_name = "goals" if position == "GARDIEN" else "players"
    path = os.path.join(path_folder, f"centiles/data_{file_name}_centiles.csv")
    df = pd.read_csv(path)
    df.rename(columns={df.columns[0]: "Player"}, inplace=True)
    df = df[df["Player"] == player_name]
    available_features = [f for f in features_centiles if f in df.columns]
    percentiles_list = df[available_features].iloc[0].tolist() if not df.empty else []
    return percentiles_list

In [14]:
def get_features_agg(position):
    if position == "GARDIEN":
        features = ['Matches', 'Minutes', 'Goals Against']
    else: 
        features = ['Matches', 'Minutes', 'Goals', 'Assists']
    return features

In [15]:
def get_data_agg(player_name, position, features_agg, path_folder):
    file_name = "goals" if position == "GARDIEN" else "players"
    path = os.path.join(path_folder, f"centiles/data_{file_name}_aggregated.csv")
    path_score = os.path.join(path_folder, f"ratings/data_{file_name}_average.csv") 
    df_score = pd.read_csv(path_score)
    df = pd.read_csv(path)
    df.rename(columns={df.columns[0]: "Player"}, inplace=True)
    df = df.merge(df_score, on=["Player", "Age", "Nationality"], how="left")
    df = df.drop(columns=[col for col in df.columns if col.endswith("_x")])
    df.columns = [col[:-2] if col.endswith('_y') else col for col in df.columns]
    df = df[df["Player"] == player_name]
    available_features = [f for f in features_agg if f in df.columns]
    agg_list = df[available_features].iloc[0].tolist() if not df.empty else []
    return agg_list

In [16]:
def update_player_info(joueur_info, agg_list, position):
    joueur_info['stats_secondaires'][0] = str(int(agg_list[0])) + " MATCHS"
    joueur_info['stats_secondaires'][1] = str(int(agg_list[1])) + " MIN"
    if position == "GARDIEN":
        joueur_info['stats_secondaires'][2] = str(int(agg_list[2])) + " BUTS PRIS"
    else: 
        joueur_info['stats_secondaires'][2] = str(int(agg_list[2] + agg_list[3])) + " G/A"

# Main

In [17]:
scraper = PlayerProfileScraper(player_name)
joueur_info = scraper.save_player_profile()

In [18]:
position = joueur_info['stats_secondaires'][5]
features_centiles, data_centiles_names = get_features_centiles(position)
percentiles = get_data_centiles(player_name, position, features_centiles, path_folder)
features_agg = get_features_agg(position)
agg_list = get_data_agg(player_name, position, features_agg, path_folder)
update_player_info(joueur_info, agg_list, position)

In [19]:
base = generate_player_canva(joueur_info, data_centiles_names, percentiles)
base.save(output_path)